# 17 - Anatomy of a PyTorch Data Pipeline

**Section:** More on Data | **Prereqs:** `FNNs/MNIST.ipynb` | **Next:** `Cross Validation/CrossValidation.ipynb`

A slow read of the two objects every notebook so far has used without
explanation.

- **`Dataset`** answers two questions: how many samples are there (`__len__`),
  and what is sample `i` (`__getitem__`). `TensorDataset` is the trivial
  implementation that just indexes tensors you already have in memory.
- **`DataLoader`** wraps a Dataset and handles batching, shuffling, parallel
  loading and collation. It is an *iterable*, not a list: it produces batches
  on demand rather than storing them.

**Why the separation.** ImageNet does not fit in RAM. A Dataset can read sample
`i` from disk on demand, and the DataLoader keeps worker processes busy fetching
the next batch while the GPU works on the current one. For small in-memory data
this is overkill - but the same code then scales without a rewrite.

Writing a custom Dataset is covered in
`Convolution and transformations/Creating_Custom_DataLoaders.ipynb`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset,DataLoader

from torchvision.transforms import ToTensor
from torchvision.datasets import MNIST


In [ ]:
# 100 samples x 20 features of Gaussian noise. Content does not matter here -
# the plumbing does.
data = np.random.randn(100,20)
data

array([[ 0.90763054,  0.03838683, -0.09190151, ..., -1.09851972,
         1.32356713, -1.42535955],
       [ 0.07496185,  0.50715212, -0.74092413, ..., -0.33985928,
         0.46695947, -1.31883339],
       [ 0.37459933, -0.57479644, -1.18711415, ...,  0.27959186,
         0.22133942, -0.63599644],
       ...,
       [-0.61297741, -0.56390012, -0.58978267, ...,  1.76001179,
        -0.04195112, -0.17808163],
       [-0.91025592,  1.43528718, -2.34900429, ..., -0.74515124,
         0.2240979 , -0.09425693],
       [-1.38346245,  0.47633124,  0.91066936, ...,  1.73500783,
        -0.41900869, -1.01725184]])

In [ ]:
# numpy -> torch. Note the dtype stays float64; PyTorch layers want float32,
# which is why .float() appears everywhere in the other notebooks.
dataT = torch.tensor(data)

In [ ]:
# TensorDataset pairs tensors by their FIRST dimension (the sample axis).
# DataLoader with no batch_size defaults to 1 - one sample per iteration.
dataset = TensorDataset(dataT)

data_loader = DataLoader(dataset)

In [ ]:
dataset

In [ ]:
data_loader

In [ ]:
# .tensors gives back the raw tensors the dataset wraps. Useful for shape
# checks; it is also how the test loader's batch size is set in other notebooks.
# Checking tensors from TensorDataset

dataset.tensors

(tensor([[ 0.9076,  0.0384, -0.0919,  ..., -1.0985,  1.3236, -1.4254],
         [ 0.0750,  0.5072, -0.7409,  ..., -0.3399,  0.4670, -1.3188],
         [ 0.3746, -0.5748, -1.1871,  ...,  0.2796,  0.2213, -0.6360],
         ...,
         [-0.6130, -0.5639, -0.5898,  ...,  1.7600, -0.0420, -0.1781],
         [-0.9103,  1.4353, -2.3490,  ..., -0.7452,  0.2241, -0.0943],
         [-1.3835,  0.4763,  0.9107,  ...,  1.7350, -0.4190, -1.0173]],
        dtype=torch.float64),)

In [ ]:
# A short aside on shapes. np.ceil turns the ramp into repeated integers -
# a quick way to fabricate class labels.
labels = np.ceil(np.linspace(0.4,4,10))
labels.shape

(10,)

In [ ]:
# (10,) -> (10,1). Labels for CrossEntropyLoss should stay 1-D; this reshape is
# here to show the difference, not as a recommendation.
labels = labels.reshape(10,1)

In [ ]:
labels

array([[1.],
       [1.],
       [2.],
       [2.],
       [2.],
       [3.],
       [3.],
       [4.],
       [4.],
       [4.]])

In [ ]:
labels.shape

(10, 1)

In [ ]:
# Transposing a column vector gives (1,10). Shape bugs are the most common
# PyTorch error, so check them explicitly whenever something breaks.
labels.T.shape

(1, 10)

In [ ]:
# The loader keeps a reference to its dataset - loader.dataset.tensors is the
# path back to the underlying data.
# Checking data loaders
data_loader.dataset.tensors

(tensor([[ 0.9076,  0.0384, -0.0919,  ..., -1.0985,  1.3236, -1.4254],
         [ 0.0750,  0.5072, -0.7409,  ..., -0.3399,  0.4670, -1.3188],
         [ 0.3746, -0.5748, -1.1871,  ...,  0.2796,  0.2213, -0.6360],
         ...,
         [-0.6130, -0.5639, -0.5898,  ...,  1.7600, -0.0420, -0.1781],
         [-0.9103,  1.4353, -2.3490,  ..., -0.7452,  0.2241, -0.0943],
         [-1.3835,  0.4763,  0.9107,  ...,  1.7350, -0.4190, -1.0173]],
        dtype=torch.float64),)

In [ ]:
# iter() starts the iteration, next() pulls one batch. This is the idiom used
# throughout the course to grab the whole test set in one go.
# Note the batch is a LIST of tensors, one per tensor in the dataset - here just
# [x], since no labels were provided.
# Only one batch

X = next(iter(data_loader))

In [ ]:
X

[tensor([[ 0.9076,  0.0384, -0.0919, -0.4787,  0.3905, -0.5001,  0.5004,  1.1301,
           1.1805,  0.5295,  0.3643,  0.3194, -0.6610, -0.5837,  0.3401, -0.5635,
           1.6457, -1.0985,  1.3236, -1.4254]], dtype=torch.float64)]